In [1]:
import os
import json
from PIL import Image
import pandas as pd
from tqdm import tqdm

# --- 1. 설정 ---
print("설정을 시작합니다...")

# VQAv2와 동일한 폴더 구조 생성
base_dir = "VMCBench_from_CSV_as_Official_VQAv2"
val_image_dir = os.path.join(base_dir, "val2014") # train.csv -> val2014
test_image_dir = os.path.join(base_dir, "test2015") # test.csv -> test2015
annotation_dir = os.path.join(base_dir, "vqa")

os.makedirs(val_image_dir, exist_ok=True)
os.makedirs(test_image_dir, exist_ok=True)
os.makedirs(annotation_dir, exist_ok=True)

# CSV 파일 경로 (현재 스크립트 실행 위치 기준)
train_csv_path = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/data_2/train.csv"
test_csv_path = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/data_2/test.csv"

# 이미지 베이스 경로 (csv 파일 내의 상대 경로를 이 경로와 결합)
train_image_base_dir = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/data_2/" # TRAIN_000.jpg -> ./train_input_images/TRAIN_000.jpg
test_image_base_dir = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/data_2/" # TEST_000.jpg -> ./test_input_images/TEST_000.jpg

# --- 2. 데이터 로드 및 전처리 함수 ---
def load_and_preprocess_data(csv_path, image_base_path, is_train_split=True):
    """CSV 파일을 로드하고 이미지 경로를 정규화하여 데이터프레임 반환"""
    print(f"CSV 파일 로드: {csv_path}")
    df = pd.read_csv(csv_path)
    
    # 이미지 경로 정규화 (./train_input_images/TRAIN_000.jpg -> /absolute/path/to/train_input_images/TRAIN_000.jpg)
    df['full_img_path'] = df['img_path'].apply(lambda x: os.path.join(image_base_path, x.lstrip('./')))
    
    # 'ID' 컬럼에서 숫자 부분만 추출하여 int로 변환 (image_id로 사용)
    # 예: 'TRAIN_000' -> 0, 'TEST_000' -> 0
    df['numeric_id'] = df['ID'].apply(lambda x: int(x.split('_')[-1]))

    return df

# --- 3. 데이터 변환 및 저장 함수 ---
def process_and_save_split(dataframe, vqav2_split_name, image_save_dir, has_answers=True):
    """
    주어진 데이터프레임을 VQAv2 형식으로 변환하고 이미지 및 JSON 파일을 저장합니다.
    Args:
        dataframe (pd.DataFrame): 변환할 데이터프레임
        vqav2_split_name (str): 'val2014' 또는 'test2015'와 같은 VQAv2 스플릿 이름
        image_save_dir (str): 이미지를 저장할 디렉토리
        has_answers (bool): 'answer' 컬럼이 존재하는지 여부 (train/val용)
    """
    print(f"'{vqav2_split_name}' 스플릿을 처리 중입니다...")
    
    questions_list = []
    annotations_list = [] # 답변이 없는 test 스플릿의 경우 비어있을 수 있음
    
    # 'A', 'B', 'C', 'D'를 0, 1, 2, 3으로 매핑 (정답 레이블용)
    choice_char_to_index = {'A': 0, 'B': 1, 'C': 2, 'D': 3}

    for idx, row in tqdm(dataframe.iterrows(), total=len(dataframe), desc=f"Processing {vqav2_split_name}"):
        image_id = row['numeric_id']
        original_id = row['ID'] # 원본 ID (예: TRAIN_000)
        
        # --- 이미지 처리 ---
        original_image_path = row['full_img_path']
        image_filename = f"COCO_{vqav2_split_name}_{str(image_id).zfill(12)}.jpg"
        image_path = os.path.join(image_save_dir, image_filename)
        
        try:
            # PIL로 이미지 로드 후 저장
            with Image.open(original_image_path) as img:
                img.save(image_path)
        except FileNotFoundError:
            print(f"경고: 이미지 파일을 찾을 수 없습니다: {original_image_path}. 이 샘플을 건너뜁니다 (ID: {original_id}).")
            continue
        except Exception as e:
            print(f"경고: 이미지 {original_image_path} 처리 중 오류 발생: {e}. 이 샘플을 건너뜁니다 (ID: {original_id}).")
            continue

        # --- 질문(Question) JSON 데이터 생성 ---
        question_text_base = row['Question']
        choices_text = [
            f"A. {row['A']}",
            f"B. {row['B']}",
            f"C. {row['C']}",
            f"D. {row['D']}"
        ]
        
        joined_choices = '\n'.join(choices_text)
        full_question_text = f"{question_text_base}\n{joined_choices}"
        
        question_data = {
            "image_id": image_id,
            "question": full_question_text,
            "question_id": image_id 
        }
        questions_list.append(question_data)

        # --- 정답(Annotation) JSON 데이터 생성 (답변이 있는 경우만) ---
        if has_answers:
            answer_char = row['answer'] # 'A', 'B', 'C', 'D'
            
            # 선택지 텍스트에서 실제 정답 텍스트 추출 (A, B, C, D 컬럼에서 해당 값을 가져옴)
            correct_answer_text = row[answer_char]
            
            answer_numeric_label = choice_char_to_index.get(answer_char)
            if answer_numeric_label is None:
                print(f"경고: 알 수 없는 정답 문자 '{answer_char}' (original_id: {original_id}). 이 샘플의 어노테이션을 건너뜁니다.")
                continue 

            annotation_data = {
                "question_type": "multiple choice",
                "multiple_choice_answer": correct_answer_text,
                "answers": [ # VQAv2는 여러 답변을 가질 수 있으므로, 10개로 복제
                    {
                        "answer": correct_answer_text,
                        "answer_confidence": "yes",
                        "question_id": image_id,
                        "image_id": image_id
                    }
                ] * 10, 
                "image_id": image_id,
                "answer_type": "other",
                "question_id": image_id,
                "answer_label": answer_numeric_label # 중요: 0, 1, 2, 3 형태의 레이블 추가
            }
            annotations_list.append(annotation_data)
            
    # --- JSON 파일 저장 ---
    # VQAv2 Question 파일 형식
    question_filename_prefix = "OpenEnded" if has_answers else "test-dev2015" # Test split might use test-dev2015
    
    final_questions_json = {
        "info": {"description": f"VMCBench converted to VQAv2 Questions for {vqav2_split_name}"},
        "task_type": "Open-Ended",
        "data_type": f"mscoco_{vqav2_split_name}",
        "questions": questions_list
    }
    question_json_path = os.path.join(annotation_dir, f"v2_{question_filename_prefix}_mscoco_{vqav2_split_name}_questions.json")
    with open(question_json_path, 'w', encoding='utf-8') as f:
        json.dump(final_questions_json, f, indent=4, ensure_ascii=False)

    # VQAv2 Annotation 파일 형식 (답변이 있는 경우만 저장)
    if has_answers:
        final_annotations_json = {
            "info": {"description": f"VMCBench converted to VQAv2 Annotations for {vqav2_split_name}"},
            "data_type": f"mscoco_{vqav2_split_name}",
            "annotations": annotations_list
        }
        annotation_json_path = os.path.join(annotation_dir, f"v2_mscoco_{vqav2_split_name}_annotations.json")
        with open(annotation_json_path, 'w', encoding='utf-8') as f:
            json.dump(final_annotations_json, f, indent=4, ensure_ascii=False)
    else:
        print(f"'{vqav2_split_name}' 스플릿은 답변이 없어 어노테이션 JSON 파일을 생성하지 않습니다.")


# --- 메인 실행 ---
# train.csv를 validation 스플릿으로 사용
print("\n--- train.csv를 VQAv2 val2014 스플릿으로 변환 중 ---")
train_df = load_and_preprocess_data(train_csv_path, train_image_base_dir, is_train_split=True)
process_and_save_split(train_df, "val2014", val_image_dir, has_answers=True)

# test.csv를 test 스플릿으로 사용
print("\n--- test.csv를 VQAv2 test2015 스플릿으로 변환 중 ---")
test_df = load_and_preprocess_data(test_csv_path, test_image_base_dir, is_train_split=False)
# 테스트 데이터셋에는 정답이 없으므로 has_answers=False로 설정
process_and_save_split(test_df, "test2015", test_image_dir, has_answers=False)


print("\n--- ✅ 데이터 변환 및 저장 완료 ---")
print("생성된 파일 구조:")
print(f"/{base_dir}")
print(f"├── /val2014/ ({len(os.listdir(val_image_dir))}개 이미지)")
print(f"├── /test2015/ ({len(os.listdir(test_image_dir))}개 이미지)")
print(f"└── /vqa/")
for fname in sorted(os.listdir(annotation_dir)):
    print(f"    ├── {fname}")

설정을 시작합니다...

--- train.csv를 VQAv2 val2014 스플릿으로 변환 중 ---
CSV 파일 로드: /data/2_data_server/cv-07/dice/2025_samsung_challenge/data/data_2/train.csv
'val2014' 스플릿을 처리 중입니다...


Processing val2014: 100%|██████████| 60/60 [00:00<00:00, 83.26it/s]



--- test.csv를 VQAv2 test2015 스플릿으로 변환 중 ---
CSV 파일 로드: /data/2_data_server/cv-07/dice/2025_samsung_challenge/data/data_2/test.csv
'test2015' 스플릿을 처리 중입니다...


Processing test2015: 100%|██████████| 852/852 [00:10<00:00, 83.58it/s] 

'test2015' 스플릿은 답변이 없어 어노테이션 JSON 파일을 생성하지 않습니다.

--- ✅ 데이터 변환 및 저장 완료 ---
생성된 파일 구조:
/VMCBench_from_CSV_as_Official_VQAv2
├── /val2014/ (60개 이미지)
├── /test2015/ (852개 이미지)
└── /vqa/
    ├── v2_OpenEnded_mscoco_val2014_questions.json
    ├── v2_mscoco_val2014_annotations.json
    ├── v2_test-dev2015_mscoco_test2015_questions.json
